# Extract the exact DOI list behind the DFT/experiment overlap set

Goal: produce a concrete, deduplicated list of the source papers behind every experimental row that sits in
the cleaned DFT<->experiment composition overlap (the same overlap established in the earlier exploration
and residual notebooks: ~9 teMatDb272 formulas, ~41 ChemDataExtractor formulas). This is the exact list to
hand to institutional library access to download all of them directly -- no more guessing at "10-20 papers".

ChemDataExtractor already carries a `DOI` column per row, so that half is direct. teMatDb272 does not
obviously expose one at the sample level in the tables used so far -- this notebook actively searches every
loaded table's columns for anything reference/DOI/citation-like rather than assuming it's absent, and prints
everything it finds so nothing is silently missed.

Run top to bottom. Everything downloads live.


## 0. Setup


In [1]:
!pip install -q matminer pymatgen requests pandas

import requests
import pandas as pd
import numpy as np
import io
import re
import zipfile
import ast
pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 160)
pd.set_option('display.max_colwidth', 120)


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.4/62.4 kB 1.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.6/55.6 kB 3.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.3/5.3 MB 32.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 829.1/829.1 kB 42.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.6/62.6 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 78.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 50.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.3/35.3 MB 21.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 331.1/331.1 kB 23.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 332.3/332.3 kB 24.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 962.5/962.5 kB 46.0 MB/s eta 0:00:00
   ━━━━━━

## 1. Load the DFT/BoltzTraP formula set (only the formulas are needed here, not the full transport data)


In [2]:
from matminer.datasets import load_dataset

df_dft = load_dataset('boltztrap_mp')
print('DFT slice shape:', df_dft.shape)


Fetching boltztrap_mp.json.gz from https://ndownloader.figshare.com/files/13297511 to /usr/local/lib/python3.13/dist-packages/matminer/datasets/boltztrap_mp.json.gz


Fetching https://ndownloader.figshare.com/files/13297511 in MB: 14.020608MB [00:00, 549.67MB/s]                         


DFT slice shape: (8924, 9)


## 2. Load teMatDb272 (zip-based, as verified in the earlier notebooks) -- load EVERY table in the zip, not just the two used before


In [3]:
repo_api = 'https://api.github.com/repos/byungkiryu/teMatDb/contents/teMatDb_publication'
resp = requests.get(repo_api, timeout=30)
listing = resp.json()

zip_entry = next((f for f in listing if isinstance(listing, list) and f['name'].lower().endswith('.zip')), None)
tematdb_tables = {}
if zip_entry:
    r = requests.get(zip_entry['download_url'], timeout=120)
    zf = zipfile.ZipFile(io.BytesIO(r.content))
    for n in zf.namelist():
        if n.lower().endswith('.csv'):
            with zf.open(n) as fh:
                try:
                    tematdb_tables[n] = pd.read_csv(fh)
                except Exception as e:
                    print(n, 'FAILED to parse:', e)

print('All tables found inside the teMatDb272 zip:')
for k, v in tematdb_tables.items():
    print(f'  {k}  ->  shape {v.shape}')
    print('     columns:', v.columns.tolist())

samples_key = next((k for k in tematdb_tables if 'sample' in k.lower()), None)
collocated_key = next((k for k in tematdb_tables if 'collocated' in k.lower()), None)
df_samples = tematdb_tables.get(samples_key)
df_collocated = tematdb_tables.get(collocated_key)


All tables found inside the teMatDb272 zip:
  teMatDb272_dataset_20250515/teMatDb_collocatedTEPs.csv  ->  shape (56641, 6)
     columns: ['sample_id', 'Temperature', 'alpha', 'rho', 'kappa', 'ZT_author_declared']
  teMatDb272_dataset_20250515/teMatDb_rawTEPs.csv  ->  shape (14717, 4)
     columns: ['sample_id', 'tepname', 'Temperature', 'tepvalue']
  teMatDb272_dataset_20250515/teMatDb_samples.csv  ->  shape (272, 12)
     columns: ['sample_id', 'YEAR', 'DOI', 'figure_number_of_targetZT', 'label_of_targetZT_in_figure', 'figure_label_description', 'mat_dimension(bulk, film, 1D, 2D)', 'GROUP', 'BASEMAT', 'Composition_by_element', 'Composition_detailed', 'SINTERING']


## 3. Actively search for a DOI/reference/citation-like column anywhere in teMatDb272

Don't assume it's missing just because it wasn't in the two tables used for the residual computation --
check every table, and also flag any column whose *values* (not just name) look like DOIs (start with
`10.`) or look like a citation string (contains a 4-digit year or "et al").


In [4]:
REF_NAME_PATTERN = re.compile(r'doi|reference|ref\b|citation|paper|source|pub', re.IGNORECASE)

print('Columns whose NAME looks reference-like, in any teMatDb272 table:')
found_by_name = []
for tname, tdf in tematdb_tables.items():
    for c in tdf.columns:
        if REF_NAME_PATTERN.search(c):
            found_by_name.append((tname, c))
            print(f'  table={tname}  column={c}  example values:', tdf[c].dropna().unique()[:5])
if not found_by_name:
    print('  none found by column name')

print()
print('Columns whose VALUES look DOI-like (start with "10.") or citation-like (contain a 4-digit year), in any table:')
found_by_value = []
year_pattern = re.compile(r'(19|20)\d{2}')
for tname, tdf in tematdb_tables.items():
    for c in tdf.columns:
        if tdf[c].dtype == object:
            sample = tdf[c].dropna().astype(str).head(200)
            if sample.str.startswith('10.').any():
                found_by_value.append((tname, c, 'doi-like'))
                print(f'  table={tname}  column={c}  (DOI-like)  example values:', tdf[c].dropna().unique()[:5])
            elif sample.str.contains('et al', case=False, na=False).any() or (sample.map(lambda x: bool(year_pattern.search(x))).mean() > 0.5 and tdf[c].nunique() < len(tdf)*0.9):
                found_by_value.append((tname, c, 'citation-like'))
                print(f'  table={tname}  column={c}  (citation-like)  example values:', tdf[c].dropna().unique()[:5])
if not found_by_value:
    print('  none found by value pattern')


Columns whose NAME looks reference-like, in any teMatDb272 table:
  table=teMatDb272_dataset_20250515/teMatDb_samples.csv  column=DOI  example values: ['10.1038/NCHEM.955' '10.1038/ncomms9144' '10.1002/adma.201501030'
 '10.1126/science.1159725' '10.1038/am.2013.86']

Columns whose VALUES look DOI-like (start with "10.") or citation-like (contain a 4-digit year), in any table:
  table=teMatDb272_dataset_20250515/teMatDb_samples.csv  column=DOI  (DOI-like)  example values: ['10.1038/NCHEM.955' '10.1038/ncomms9144' '10.1002/adma.201501030'
 '10.1126/science.1159725' '10.1038/am.2013.86']


## 4. Cleaned DFT<->experiment composition overlap (same filter used throughout this project)


In [5]:
from pymatgen.core.composition import Composition

NON_FORMULA_LABELS = {'HH', 'SKD', 'M2Q', 'ABQ2', 'Cu2Q', 'clathrate', 'etc', 'CNT'}
ISOTOPE_SYMBOLS = {'D', 'T'}

def to_reduced_formula(s, exclude_labels=NON_FORMULA_LABELS):
    if not isinstance(s, str) or not s.strip():
        return None
    if s.strip() in exclude_labels:
        return None
    try:
        comp = Composition(s)
        if any(el.symbol in ISOTOPE_SYMBOLS for el in comp.elements):
            return None
        return comp.reduced_formula
    except Exception:
        return None

dft_formulas = set(df_dft['formula'].map(to_reduced_formula).dropna())

tematdb_formulas = set()
if df_samples is not None and 'BASEMAT' in df_samples.columns:
    df_samples = df_samples.copy()
    df_samples['reduced_formula'] = df_samples['BASEMAT'].map(to_reduced_formula)
    tematdb_formulas = set(df_samples['reduced_formula'].dropna())
overlap_tematdb = dft_formulas & tematdb_formulas
print('Cleaned overlap DFT <-> teMatDb272:', len(overlap_tematdb), sorted(overlap_tematdb))


/usr/lib/python3.13/functools.py:1025: UserWarning: No Pauling electronegativity for Ar. Setting to NaN. This has no physical meaning, and is mainly done to avoid errors caused by the code expecting a float.
  val = self.func(instance)


Cleaned overlap DFT <-> teMatDb272: 9 ['CoSb3', 'CuBiSeO', 'GeTe', 'In2O3', 'PbS', 'PbSe', 'SnS', 'SnSe', 'TePb']


In [6]:
fig_api = 'https://api.figshare.com/v2/articles/19658787'
resp = requests.get(fig_api, timeout=30)
article = resp.json()
files = article.get('files', [])
zip_candidates = [f for f in files if f['name'].lower().endswith('.zip')]

def load_csvs_from_zip_url(url, timeout=120):
    r = requests.get(url, timeout=timeout)
    zf = zipfile.ZipFile(io.BytesIO(r.content))
    tables = {}
    for n in zf.namelist():
        if n.lower().endswith('.csv'):
            with zf.open(n) as fh:
                try:
                    tables[n] = pd.read_csv(fh)
                except Exception as e:
                    print(n, 'FAILED to parse:', e)
    return tables

cde_tables = {}
te_db_zip = next((f for f in zip_candidates if 'TE_databases' in f['name']), None)
if te_db_zip:
    cde_tables = load_csvs_from_zip_url(te_db_zip['download_url'])
if not cde_tables:
    other_zip = next((f for f in zip_candidates if f is not te_db_zip), None)
    if other_zip:
        cde_tables = load_csvs_from_zip_url(other_zip['download_url'])

df_cde = None
if cde_tables:
    main_key = max(cde_tables, key=lambda k: cde_tables[k].shape[0])
    df_cde = cde_tables[main_key]
    print('Using', main_key, ':', df_cde.shape)

resolution_col = next((c for c in df_cde.columns if 'resolution' in c.lower()), None) if df_cde is not None else None
name_col = next((c for c in df_cde.columns if c.lower() == 'name'), None) if df_cde is not None else None
specifier_col = next((c for c in df_cde.columns if c.lower() == 'specifier'), None) if df_cde is not None else None
value_col = next((c for c in df_cde.columns if c.lower() == 'value'), None) if df_cde is not None else None
doi_col = next((c for c in df_cde.columns if c.lower() == 'doi'), None) if df_cde is not None else None
print('Key columns:', resolution_col, name_col, specifier_col, value_col, doi_col)

def extract_host(s):
    if not isinstance(s, str):
        return None
    m = re.search(r"host['\"]?\s*[:=]\s*['\"]?([A-Za-z0-9().]+)", s)
    return m.group(1) if m else None

cde_host_series = None
if df_cde is not None:
    if resolution_col and df_cde[resolution_col].notna().any():
        cde_host_series = df_cde[resolution_col].map(extract_host)
    if cde_host_series is None or cde_host_series.notna().sum() == 0:
        cde_host_series = df_cde[name_col]

cde_formulas = set(cde_host_series.map(to_reduced_formula).dropna()) if cde_host_series is not None else set()
overlap_cde = dft_formulas & cde_formulas
print()
print('Cleaned overlap DFT <-> ChemDataExtractor:', len(overlap_cde), sorted(overlap_cde)[:40])


Using TE_databases/CSV/main_tedb.csv : (22805, 24)
Key columns: Resolution Name Specifier Value DOI

Cleaned overlap DFT <-> ChemDataExtractor: 41 ['AgBi3S5', 'AlCuO2', 'AlN', 'BaCeO3', 'BaTiO3', 'Bi2O3', 'Bi2S3', 'CdSe', 'CoSb3', 'CrCuO2', 'Cu2S', 'CuBiSeO', 'CuBr', 'GaN', 'Ge', 'GeSe', 'GeTe', 'H2O2', 'In2O3', 'KGaSb4', 'MoS2', 'NaHO', 'PbS', 'PbSe', 'Sb2Se3', 'Si', 'SiC', 'SnO2', 'SnS', 'SnSe', 'Sr3Ti2O7', 'SrMnO3', 'SrTe', 'TePb', 'TiO2', 'TlInSe2', 'WO3', 'YPO4', 'Zn3In2O6', 'ZnO']


/usr/lib/python3.13/functools.py:1025: UserWarning: No Pauling electronegativity for Ts. Setting to NaN. This has no physical meaning, and is mainly done to avoid errors caused by the code expecting a float.
  val = self.func(instance)


## 5. Unique DOIs from ChemDataExtractor, restricted to the overlap set and to Seebeck-relevant rows

Restricting to rows whose `Specifier` is Seebeck-related keeps this list to papers that actually report the
quantity this thesis needs (rather than every paper that ever mentioned one of these 41 formulas for *any*
property), so the DOI list stays tight and directly useful.


In [7]:
df_cde_overlap_dois = pd.DataFrame()
if df_cde is not None and specifier_col and doi_col and overlap_cde:
    seebeck_mask = df_cde[specifier_col].astype(str).str.contains('seebeck', case=False, na=False)
    df_seebeck = df_cde[seebeck_mask].copy()
    df_seebeck['host_formula'] = cde_host_series.loc[df_seebeck.index].map(to_reduced_formula)
    df_seebeck = df_seebeck[df_seebeck['host_formula'].isin(overlap_cde)]

    df_cde_overlap_dois = (
        df_seebeck.groupby(doi_col)
        .agg(n_rows=(doi_col, 'size'), formulas=('host_formula', lambda s: sorted(set(s))))
        .reset_index()
        .rename(columns={doi_col: 'DOI'})
        .sort_values('n_rows', ascending=False)
    )
    print('Unique DOIs behind the ChemDataExtractor overlap (Seebeck rows only):', len(df_cde_overlap_dois))
    display(df_cde_overlap_dois)
else:
    print('Could not build this -- check doi_col/specifier_col/overlap_cde above')


Unique DOIs behind the ChemDataExtractor overlap (Seebeck rows only): 19


,DOI,n_rows,formulas
10,10.1016/j.mtener.2018.06.010,6,[PbSe]
12,10.1016/j.surfin.2020.100504,5,[ZnO]
18,10.1134/S1063783419100354,2,[ZnO]
2,10.1016/j.ceramint.2018.03.100,1,[PbSe]
1,10.1016/j.apsusc.2019.144066,1,[MoS2]
0,10.1007/s10853-019-04008-3,1,[Bi2S3]
3,10.1016/j.electacta.2011.04.063,1,[TiO2]
6,10.1016/j.jallcom.2019.152463,1,[TePb]
7,10.1016/j.materresbull.2013.02.004,1,[Bi2S3]
5,10.1016/j.jallcom.2017.12.249,1,[ZnO]


## 6. Unique reference identifiers from teMatDb272, restricted to the overlap set

Uses whatever reference-like column Section 3 found. If Section 3 found nothing, this cell says so plainly
and falls back to printing the raw sample metadata (GROUP/BASEMAT/sample_id) for the overlap set instead, so
there's still something concrete to search by (e.g. via Google Scholar / the paper's own supplementary data)
even without a direct DOI column.


In [8]:
df_tematdb_overlap_refs = pd.DataFrame()

# Prefer an explicit DOI-like column if Section 3 found one; otherwise fall back to any citation-like column;
# otherwise fall back to raw identifying metadata.
ref_col = None
ref_table = None
for tname, c in found_by_name:
    if 'doi' in c.lower():
        ref_table, ref_col = tname, c
        break
if ref_col is None:
    for tname, c, kind in found_by_value:
        if kind == 'doi-like':
            ref_table, ref_col = tname, c
            break
if ref_col is None and found_by_name:
    ref_table, ref_col = found_by_name[0]
if ref_col is None and found_by_value:
    ref_table, ref_col = found_by_value[0][0], found_by_value[0][1]

print('Reference column selected:', ref_table, '/', ref_col)

if ref_col is not None:
    ref_df = tematdb_tables[ref_table]
    # try to join back to df_samples via sample_id if this table isn't df_samples itself
    if ref_table == samples_key:
        merged = df_samples
    else:
        join_key = next((c for c in ref_df.columns if c in df_samples.columns and 'sample' in c.lower()), None)
        merged = df_samples.merge(ref_df, on=join_key, how='left') if join_key else None

    if merged is not None:
        overlap_rows = merged[merged['reduced_formula'].isin(overlap_tematdb)]
        df_tematdb_overlap_refs = (
            overlap_rows.groupby(ref_col)
            .agg(n_rows=(ref_col, 'size'), formulas=('reduced_formula', lambda s: sorted(set(s))))
            .reset_index()
            .sort_values('n_rows', ascending=False)
        )
        print('Unique reference values behind the teMatDb272 overlap:', len(df_tematdb_overlap_refs))
        display(df_tematdb_overlap_refs)
    else:
        print('Found a candidate reference column but could not join it back to the sample table automatically --')
        print('inspect tematdb_tables[%r] by hand.' % ref_table)
else:
    print('No DOI/reference/citation-like column found anywhere in teMatDb272.')
    print('Falling back to raw sample metadata for the overlap set -- search these by hand (formula + GROUP',
          'on Google Scholar, or check the teMatDb272 GitHub repo/paper (Ryu et al.) for a separate reference table):')
    fallback_cols = [c for c in ['sample_id', 'reduced_formula', 'GROUP', 'SINTERING'] if c in df_samples.columns]
    df_tematdb_overlap_refs = df_samples[df_samples['reduced_formula'].isin(overlap_tematdb)][fallback_cols]
    display(df_tematdb_overlap_refs)


Reference column selected: teMatDb272_dataset_20250515/teMatDb_samples.csv / DOI
Unique reference values behind the teMatDb272 overlap: 87


,DOI,n_rows,formulas
60,10.1039/c5ee02979a,2,[TePb]
68,10.1039/c9ee03897c,2,[SnSe]
79,10.1109/ICT.1997.667089,2,[GeTe]
55,10.1039/c0ee00456a,2,[TePb]
0,10.1002/adfm.201000878,1,[TePb]
...,...,...,...
82,10.1126/science.1159725,1,[TePb]
83,10.1126/science.aad3749,1,[SnSe]
84,10.1126/science.aaq1479,1,[SnSe]
85,10.1126/science.aax5123,1,[SnS]


## 7. Final combined, deduplicated DOI list -- this is what goes to institutional library access


In [9]:
final_dois = set()
if len(df_cde_overlap_dois):
    final_dois |= set(df_cde_overlap_dois['DOI'].dropna().astype(str))
if len(df_tematdb_overlap_refs) and 'DOI' in df_tematdb_overlap_refs.columns.astype(str).str.upper().tolist():
    doi_like_col = [c for c in df_tematdb_overlap_refs.columns if 'doi' in c.lower()]
    if doi_like_col:
        final_dois |= set(df_tematdb_overlap_refs[doi_like_col[0]].dropna().astype(str))

print('=== FINAL DOI LIST (ChemDataExtractor side + any confirmed teMatDb272 DOIs) ===')
print('Total unique DOIs:', len(final_dois))
for d in sorted(final_dois):
    print(' ', d)

print()
print('If Section 6 could not find a DOI column for teMatDb272, its papers are NOT included in the count')
print('above -- use the raw metadata table printed in Section 6 to look those up separately (they will add')
print('to this list, not replace it).')

pd.DataFrame({'DOI': sorted(final_dois)}).to_csv('overlap_paper_dois.csv', index=False)
print()
print('Saved to overlap_paper_dois.csv (download it from the Colab file browser on the left).')


=== FINAL DOI LIST (ChemDataExtractor side + any confirmed teMatDb272 DOIs) ===
Total unique DOIs: 106
  10.1002/adfm.201000878
  10.1002/adfm.201001307
  10.1002/adfm.201201221
  10.1002/adfm.201801617
  10.1002/adfm.201806613
  10.1002/adfm.201910059
  10.1002/adfm.202005479
  10.1002/adfm.202007340
  10.1002/adma.201103153
  10.1002/adma.201703148
  10.1002/adma.201705942
  10.1002/aelm.201500228
  10.1002/aenm.201000072
  10.1002/aenm.201400486
  10.1002/aenm.201602582
  10.1002/aenm.201800659
  10.1002/aenm.201803072
  10.1002/aenm.201803242
  10.1002/aenm.201901334
  10.1002/aenm.202002588
  10.1007/s10853-019-04008-3
  10.1007/s11664-009-0779-2
  10.1007/s11664-009-0993-y
  10.1007/s11664-013-2819-1
  10.1007/s11664-013-2832-4
  10.1007/s11664-015-4251-1
  10.1007/s13391-014-4149-8
  10.1016/j.apsusc.2019.144066
  10.1016/j.ceramint.2018.03.100
  10.1016/j.electacta.2011.04.063
  10.1016/j.elspec.2009.03.001
  10.1016/j.intermet.2013.04.018
  10.1016/j.jallcom.2012.08.041
  10.1